In [15]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier
import lightgbm as lgb
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix
from transformers import BertTokenizer, BertModel
import torch
from tqdm import tqdm, trange
from sklearn.model_selection import StratifiedShuffleSplit

In [3]:
train_df = pd.read_csv('/content/training_data_binary.csv')
test_df = pd.read_csv('/content/testing_data_binary.csv')
full_df = pd.concat([train_df, test_df], ignore_index=True)

In [4]:
X_text = full_df['prompt']
y = full_df['jailbreak'].values

In [5]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BertModel.from_pretrained('bert-base-uncased')

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

In [7]:
def extract_bert_features(texts, tokenizer, model, max_length=128):
    embeddings = []
    for text in tqdm(texts, desc="Extracting BERT Features"):
        tokens = tokenizer.encode_plus(
            text,
            max_length=max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        with torch.no_grad():
            output = model(
                input_ids=tokens['input_ids'],
                attention_mask=tokens['attention_mask']
            )
            embedding = output.last_hidden_state[:, 0, :].squeeze().numpy()
            embeddings.append(embedding)
    return np.array(embeddings)

In [8]:
X_bert = extract_bert_features(X_text, tokenizer, bert_model)

Extracting BERT Features: 100%|██████████| 6129/6129 [55:15<00:00,  1.85it/s]


In [9]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Extra Trees Classifier": ExtraTreesClassifier(n_estimators=100),
    "LightGBM": lgb.LGBMClassifier()
}

In [10]:
metrics_dict = {
    model_name: {"Accuracy": [], "AUC": [], "TPR": [], "FNR": []}
    for model_name in models
}

In [11]:
NUM_RUNS = 30
sss = StratifiedShuffleSplit(n_splits=NUM_RUNS, test_size=0.2, random_state=42)

In [16]:
for split_idx, (train_index, test_index) in zip(trange(NUM_RUNS, desc="Running 30 Random Splits"), sss.split(X_bert, y)):
    X_train, X_test = X_bert[train_index], X_bert[test_index]
    y_train, y_test = y[train_index], y[test_index]

    for model_name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # Use predict_proba for AUC if available, else fallback to decision_function
        try:
            y_prob = model.predict_proba(X_test)[:, 1]
        except:
            y_prob = model.decision_function(X_test)

        cm = confusion_matrix(y_test, y_pred)
        tn, fp, fn, tp = cm.ravel()

        acc = accuracy_score(y_test, y_pred)
        auc = roc_auc_score(y_test, y_prob)
        tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
        fnr = fn / (tp + fn) if (tp + fn) > 0 else 0

        metrics_dict[model_name]["Accuracy"].append(acc)
        metrics_dict[model_name]["AUC"].append(auc)
        metrics_dict[model_name]["TPR"].append(tpr)
        metrics_dict[model_name]["FNR"].append(fnr)

Running 30 Random Splits:   0%|          | 0/30 [00:00<?, ?it/s]/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


[LightGBM] [Info] Number of positive: 2024, number of negative: 2879
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.111618 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 195840
[LightGBM] [Info] Number of data points in the train set: 4903, number of used features: 768
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.412808 -> initscore=-0.352367
[LightGBM] [Info] Start training from score -0.352367


/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
Running 30 Random Splits:   3%|▎         | 1/30 [00:51<24:43, 51.16s/it]/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


[LightGBM] [Info] Number of positive: 2024, number of negative: 2879
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.055973 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 195840
[LightGBM] [Info] Number of data points in the train set: 4903, number of used features: 768
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.412808 -> initscore=-0.352367
[LightGBM] [Info] Start training from score -0.352367


/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
Running 30 Random Splits:   7%|▋         | 2/30 [01:17<17:08, 36.73s/it]/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


[LightGBM] [Info] Number of positive: 2024, number of negative: 2879
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.057928 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 195840
[LightGBM] [Info] Number of data points in the train set: 4903, number of used features: 768
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.412808 -> initscore=-0.352367
[LightGBM] [Info] Start training from score -0.352367


... (51 outputs left out)

[LightGBM] [Info] Number of positive: 2024, number of negative: 2879
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.065463 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 195840
[LightGBM] [Info] Number of data points in the train set: 4903, number of used features: 768
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.412808 -> initscore=-0.352367
[LightGBM] [Info] Start training from score -0.352367


/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
Running 30 Random Splits:  97%|█████████▋| 29/30 [12:51<00:25, 25.19s/it]/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


[LightGBM] [Info] Number of positive: 2024, number of negative: 2879
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.063435 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 195840
[LightGBM] [Info] Number of data points in the train set: 4903, number of used features: 768
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.412808 -> initscore=-0.352367
[LightGBM] [Info] Start training from score -0.352367


/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
Running 30 Random Splits: 100%|██████████| 30/30 [13:15<00:00, 26.53s/it]


In [17]:
summary = []
for model_name, metrics in metrics_dict.items():
    summary.append({
        "Model": model_name,
        "Accuracy Mean": round(np.mean(metrics["Accuracy"]), 4),
        "Accuracy Std": round(np.std(metrics["Accuracy"]), 4),
        "AUC Mean": round(np.mean(metrics["AUC"]), 4),
        "AUC Std": round(np.std(metrics["AUC"]), 4),
        "TPR Mean": round(np.mean(metrics["TPR"]), 4),
        "TPR Std": round(np.std(metrics["TPR"]), 4),
        "FNR Mean": round(np.mean(metrics["FNR"]), 4),
        "FNR Std": round(np.std(metrics["FNR"]), 4),
    })

summary_df = pd.DataFrame(summary)
print("\n=== Final Results After 30 Random Splits ===")
print(summary_df)


=== Final Results After 30 Random Splits ===
                    Model  Accuracy Mean  Accuracy Std  AUC Mean  AUC Std  \
0     Logistic Regression         0.8745        0.0081    0.9397   0.0054   
1  Extra Trees Classifier         0.8805        0.0080    0.9482   0.0049   
2                LightGBM         0.8862        0.0090    0.9526   0.0047   

   TPR Mean  TPR Std  FNR Mean  FNR Std  
0    0.8540   0.0118    0.1460   0.0118  
1    0.8218   0.0177    0.1782   0.0177  
2    0.8545   0.0152    0.1455   0.0152  
